In [7]:
import dill
try:
    dill.load_session('session2.pkl')
    print('Session restored successfully ✅')
except FileNotFoundError:
    print('No saved session found. Run the notebook once and it will be saved.')

Session restored successfully ✅


In [1]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


In [2]:
# Cell 2: Load & combine CSV files
data_dir = r"C:\Users\Riya\Documents\heritage-predictor\env\Data"
dfs = []

for file in os.listdir(data_dir):
    if file.endswith(".csv"):
        city = file.replace(".csv", "")
        df = pd.read_csv(os.path.join(data_dir, file))
        df['City'] = city
        dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data["Datetime"] = pd.to_datetime(data["Datetime"], dayfirst=True, errors="coerce")
data = data.sort_values(['City', 'Datetime']).reset_index(drop=True)


In [3]:
# Cell 3: Feature engineering

# Ensure 'Datetime' is actually a datetime type
data['Datetime'] = pd.to_datetime(data['Datetime'], dayfirst=True, errors='coerce')


# Time features
data['hour'] = data['Datetime'].dt.hour
data['dayofyear'] = data['Datetime'].dt.dayofyear
data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
data['day_sin'] = np.sin(2 * np.pi * data['dayofyear'] / 365)
data['day_cos'] = np.cos(2 * np.pi * data['dayofyear'] / 365)

# Lag features
for lag in [1, 2, 3, 6, 12, 24]:
    data[f'outdoor_temp_lag_{lag}'] = data['Outdoor_Temp'].shift(lag)
    data[f'indoor_temp_lag_{lag}'] = data['Indoor_Temp'].shift(lag)
    data[f'outdoor_hum_lag_{lag}'] = data['Outdoor_Humidity'].shift(lag)
    data[f'indoor_hum_lag_{lag}'] = data['Indoor_Humidity'].shift(lag)

# Rolling means
for window in [3, 6, 12, 24]:
    data[f'outdoor_temp_roll_{window}'] = data['Outdoor_Temp'].rolling(window).mean()
    data[f'indoor_temp_roll_{window}'] = data['Indoor_Temp'].rolling(window).mean()
    data[f'outdoor_hum_roll_{window}'] = data['Outdoor_Humidity'].rolling(window).mean()
    data[f'indoor_hum_roll_{window}'] = data['Indoor_Humidity'].rolling(window).mean()

# Drop rows with NaNs from lag/rolling
data = data.dropna().reset_index(drop=True)

print("Feature engineering complete. Shape:", data.shape)


Feature engineering complete. Shape: (31526, 52)


In [4]:
# Cell 4: Prepare data and create sequences for LSTM with temporal dependencies

data = pd.get_dummies(data, columns=['City'], drop_first=True)

features = [col for col in data.columns if col not in ['Datetime', 'Indoor_Temp', 'Indoor_Humidity']]

X = data[features].values
y = data[['Indoor_Temp', 'Indoor_Humidity']].values

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Train/test split before creating sequences
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, shuffle=False)

# Function to create sequences
def create_sequences(X, y, seq_length=24):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])
    return np.array(X_seq), np.array(y_seq)

seq_length = 48  # e.g., 24 timesteps (hours)

X_train_seq, y_train_seq = create_sequences(X_train, y_train, seq_length)
X_test_seq, y_test_seq = create_sequences(X_test, y_test, seq_length)

print("X_train_seq shape:", X_train_seq.shape)
print("y_train_seq shape:", y_train_seq.shape)


X_train_seq shape: (25172, 48, 51)
y_train_seq shape: (25172, 2)


In [5]:
# Cell 5: Define & train LSTM model with sequences

model = Sequential()
model.add(LSTM(64, activation='tanh', input_shape=(seq_length, X_train_seq.shape[2])))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dense(2))  # two outputs: Indoor_Temp & Indoor_Humidity

model.compile(optimizer='adam', loss='mse')

history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_test_seq, y_test_seq),
    epochs=20,
    batch_size=32,
    verbose=1
)


c:\Users\Riya\Documents\heritage-predictor\env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 0.0779 - val_loss: 0.1377
Epoch 2/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0337 - val_loss: 0.0881
Epoch 3/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 0.0263 - val_loss: 0.0735
Epoch 4/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 0.0228 - val_loss: 0.0674
Epoch 5/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0202 - val_loss: 0.0625
Epoch 6/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0185 - val_loss: 0.0543
Epoch 7/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0170 - val_loss: 0.0541
Epoch 8/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0155 - val_loss: 0.0463
Epoch 9/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0144 - val_loss: 0.0447
Epoch 10/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0135 - val_loss: 0.0430
Epoch 11/20
787/787 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0130 - val_loss: 0.0345
Epoch 12/20
787/787 ━━━━━━━━━━━━━━━━━━━━

In [15]:
# Cell 6: Evaluation with sequence inputs

y_pred_scaled = model.predict(X_test_seq)
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_test_inv = scaler_y.inverse_transform(y_test_seq)

target = ['Indoor_Temp', 'Indoor_Humidity']
for i, col in enumerate(target):
    mae = mean_absolute_error(y_test_inv[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_test_inv[:, i], y_pred[:, i]))
    r2 = r2_score(y_test_inv[:, i], y_pred[:, i])
    print(f"{col} - MAE: {mae:.3f}, RMSE: {rmse:.3f}, R²: {r2:.3f}")


196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Indoor_Temp - MAE: 0.757, RMSE: 1.055, R²: 0.939
Indoor_Humidity - MAE: 2.169, RMSE: 3.555, R²: 0.931


In [6]:
import dill
dill.dump_session('session2.pkl')
print('Session saved successfully ✅')

Session saved successfully ✅
